# 🎙️ Hindi TTS Dataset Pipeline
**GPU-accelerated pipeline on Google Colab**

Steps:
1. Setup & Install
2. Download YouTube Audio
3. Vocal Separation (Demucs)
4. Speaker Diarization (Pyannote 3.1)
5. Transcription (Whisper large-v3)
6. LLM Auto-tagging (Groq)
7. Export Manifest


## ✅ Step 0: Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
!nvidia-smi

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q yt-dlp demucs pyannote.audio openai-whisper pandas tqdm openai python-dotenv soundfile

## 🔑 Step 2: Set API Keys

In [ ]:
import os

# Hugging Face token (for Pyannote)
# Get from: https://huggingface.co/settings/tokens
os.environ['HF_TOKEN'] = 'hf_your_token_here'  # <-- apna token daalo

# Groq API key (for LLM tagging)
# Get from: https://console.groq.com/
os.environ['GROQ_API_KEY'] = 'gsk_your_key_here'  # <-- apni key daalo

print('[+] Keys set!')

## 📁 Step 3: Clone GitHub Repo & Setup Folders

In [ ]:
# Apna GitHub repo URL yahan daalo
REPO_URL = 'https://github.com/YOUR_USERNAME/hindi-tts-dataset.git'

!git clone {REPO_URL}
%cd hindi-tts-dataset

# Data folders create karo
!mkdir -p data/01_raw_audio data/02_isolated_vocals data/03_diarized_segments data/04_transcripts
print('[+] Setup complete!')

## 🎵 Step 4: Download YouTube Audio

In [ ]:
%cd /content/hindi-tts-dataset
!python3 scripts/01_scrape_youtube.py

## 🎤 Step 5: Vocal Separation (Demucs)

In [ ]:
%cd /content/hindi-tts-dataset
!python3 scripts/02_separate_audio.py

## 👥 Step 6: Speaker Diarization (Pyannote 3.1)

In [ ]:
import os
# Inject token into env for pyannote
os.environ['HF_TOKEN'] = os.environ.get('HF_TOKEN', '')

%cd /content/hindi-tts-dataset

# Write token to .env for scripts
with open('.env', 'w') as f:
    f.write(f"HF_TOKEN={os.environ['HF_TOKEN']}\n")
    f.write(f"GROQ_API_KEY={os.environ['GROQ_API_KEY']}\n")

!python3 scripts/03_diarize_pyannote.py

## 📝 Step 7: Transcription (Whisper large-v3)

In [ ]:
%cd /content/hindi-tts-dataset
!python3 scripts/04_transcribe_whisper.py

## 🏷️ Step 8: LLM Auto-tagging (Groq)

In [ ]:
%cd /content/hindi-tts-dataset
!python3 scripts/05_llm_autotagging.py

## 📦 Step 9: Export Final Manifest

In [ ]:
%cd /content/hindi-tts-dataset
!python3 scripts/06_export_manifest.py

## 💾 Step 10: Download Output to Local Machine

In [ ]:
from google.colab import files
import shutil, os

# Zip the processed data
shutil.make_archive('/content/hindi_tts_output', 'zip', '/content/hindi-tts-dataset/data')
files.download('/content/hindi_tts_output.zip')
print('[+] Download started!')

## 🤖 Step 11: Orchestrator Demo (Agent 01 + 02)

In [ ]:
%cd /content/hindi-tts-dataset
!python3 scripts/07_orchestrator_poc.py